# Renown Combat Lab — Reorganized

End-to-end combat analytics for *Renown*. Structured to compute once, analyze many times.

**Section 1 (Compute)** runs the heavy simulations and writes CSVs to disk:
- Main tournament (Random vs Random) with first-skirmish tactic pair logging
- Playstyle-aware tournament (each loadout uses its default playstyle)
- Forced-tactics matrix sweep on a stratified sample
- Horde-mode survival on a stratified sample

All later sections read those CSVs — no re-running sims to look at a new angle.

**Section 2+** organizes the analysis by:
- Pursuit / Domain — does spending correlate with winning?
- Retinue performance
- Equipment (weapons, shields, armor, bastard, lance)
- Tactics (empirical matrix from main run + forced-matrix sweep)
- Playstyles (style-aware tournament results)
- Horde Mode
- Per-loadout rollups (durability, kill power, robustness, counters, casualty sources)

## 0. Setup — imports and global config

Change `LOADOUT_SOURCE` to `"csv"` to load a pre-computed pool from `loadouts.csv` (faster startup, reproducible) or `"generator"` to regenerate from `archetype_pool()`.

Tournament sizes: `N_RUNS_MAIN` is the most expensive (n_loadouts² × N_RUNS battles). `N_RUNS_PLAYSTYLE` runs a second tournament with style-aware loadouts. `N_RUNS_FORCED_TACTICS` runs a smaller forced-tactics sweep for the head-to-head tactic matrix.

In [ ]:
import os, sys, time, json, random, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

# Project modules
import renown_combat, vectorized_combat, loadouts, playstyles, tournament_vec, analysis
import tactics_analysis
for m in [renown_combat, vectorized_combat, loadouts, playstyles, tournament_vec, analysis, tactics_analysis]:
    importlib.reload(m)
vectorized_combat.invalidate_tactic_tables()

from renown_combat import TACTICS

# ── CONFIG ─────────────────────────────────────────────────────────────
LOADOUT_SOURCE = "csv"           # "csv" or "generator"
LOADOUT_CSV    = "loadouts.csv"
OUT_DIR        = "lab_out"
os.makedirs(OUT_DIR, exist_ok=True)

# Main Random-vs-Random tournament — the primary data source
N_RUNS_MAIN    = 100
N_WORKERS_MAIN = max(1, (os.cpu_count() or 4) - 1)

# Playstyle-aware tournament — each loadout uses its assigned default playstyle
N_RUNS_PLAYSTYLE = 100

# Forced-tactics matrix on a stratified sample
N_RUNS_FORCED  = 100
FORCED_SAMPLE_SIZE = 60          # stratified across retinue × MPC buckets

# Horde mode (multi-battle survival)
N_RUNS_HORDE   = 50
HORDE_BATTLES  = 8
HORDE_SAMPLE_SIZE = 60

# Reproducibility
random.seed(2026)
np.random.seed(2026)

## 0.1 Load loadout pool

In [ ]:
if LOADOUT_SOURCE == "csv" and os.path.exists(LOADOUT_CSV):
    pool = loadouts.archetype_pool(csv_path=LOADOUT_CSV)
    print(f"Loaded {len(pool)} loadouts from {LOADOUT_CSV}")
else:
    pool = loadouts.archetype_pool()
    print(f"Generated {len(pool)} loadouts from archetype_pool()")
    if LOADOUT_SOURCE == "csv":
        print(f"  ({LOADOUT_CSV} not found — falling back to generator)")

# Quick summary
print("\nBy retinue:")
for r, n in Counter(ld.retinue for ld in pool).most_common():
    print(f"  {r:<15} {n}")
print(f"\nBy MPC (military pursuit count):")
mpc_counts = Counter(ld.military_pursuit_count for ld in pool)
for mpc in sorted(mpc_counts):
    print(f"  mpc={mpc:<3} {mpc_counts[mpc]}")

# 1. Compute — run all simulations once

Each subsection writes CSVs to `OUT_DIR`. Skip subsections you've already run if the CSVs are on disk.

## 1A. Main tournament (Random vs Random)

This is the canonical data source. Every loadout plays every other loadout `N_RUNS_MAIN` times with Random playstyles on both sides. Outputs:
- `summary.csv` — one row per loadout with aggregated metrics
- `matchups.csv` — one row per (a_loadout × b_loadout) pair
- `tactic_matrix.csv` — empirical 7×7 tactic-pair matrix (NEW)

The `tactic_matrix.csv` is produced for free from the main tournament — every first-skirmish tactic pair is logged and attributed to final outcomes. This means we don't need a separate forced-tactic sweep to see how tactics interact in real (random) play.

In [ ]:
MAIN_SUMMARY = os.path.join(OUT_DIR, "summary.csv")
MAIN_MATCHUPS = os.path.join(OUT_DIR, "matchups.csv")
MAIN_TACTIC_MATRIX = os.path.join(OUT_DIR, "tactic_matrix.csv")

if os.path.exists(MAIN_SUMMARY) and os.path.exists(MAIN_MATCHUPS):
    print(f"SKIP — main tournament CSVs already exist in {OUT_DIR}")
    print(f"  Delete those files and re-run this cell to recompute.")
else:
    t0 = time.time()
    tournament_vec.run_tournament_vec(
        pool, n_runs=N_RUNS_MAIN, output_dir=OUT_DIR,
        filename_suffix="", n_workers=N_WORKERS_MAIN,
        verbose=True, print_every=max(1, len(pool)//20))
    print(f"\nMain tournament complete in {time.time()-t0:.0f}s")

## 1B. Playstyle-aware tournament

Each loadout is assigned its theoretically-optimal playstyle by `assign_default_playstyle()`, then the tournament reruns with everyone using their assigned style instead of Random. This is the comparison set for evaluating playstyle quality.

In [ ]:
PLAYSTYLE_SUMMARY = os.path.join(OUT_DIR, "summary_playstyle.csv")
PLAYSTYLE_MATCHUPS = os.path.join(OUT_DIR, "matchups_playstyle.csv")
PLAYSTYLE_TACTIC_MATRIX = os.path.join(OUT_DIR, "tactic_matrix_playstyle.csv")

if os.path.exists(PLAYSTYLE_SUMMARY) and os.path.exists(PLAYSTYLE_MATCHUPS):
    print(f"SKIP — playstyle tournament CSVs already exist in {OUT_DIR}")
else:
    # Assign defaults — mutate pool in place
    style_pool = []
    for ld in pool:
        ps = playstyles.assign_default_playstyle(ld)
        style_pool.append(ld._replace(playstyle=ps))

    # KT-twins experiment: for each KT loadout (assigned Unshakable above), also add
    # a twin with the SAME retinue+equipment but the equipment-natural NON-Unshakable
    # playstyle. This isolates the playstyle effect within KT — does KT win because
    # Unshakable is good, or because KT's rout-immunity stat carries every playstyle?
    kt_twins = loadouts.kt_twins(pool)
    style_pool.extend(kt_twins)
    print(f"Added {len(kt_twins)} KT-twin loadouts (same KT, equipment-natural playstyle)")

    print(f"Playstyle distribution across {len(style_pool)} loadouts:")
    for ps, n in Counter(ld.playstyle for ld in style_pool).most_common():
        print(f"  {ps:<14} {n}")
    print()

    t0 = time.time()
    tournament_vec.run_tournament_vec(
        style_pool, n_runs=N_RUNS_PLAYSTYLE, output_dir=OUT_DIR,
        filename_suffix="_playstyle", n_workers=N_WORKERS_MAIN,
        verbose=True, print_every=max(1, len(style_pool)//20))
    print(f"\nPlaystyle tournament complete in {time.time()-t0:.0f}s")

## 1C. Forced-tactics matrix (stratified sample)

For a smaller stratified sample (~60 loadouts), force every (a_tactic, b_tactic) pair and measure win rates. This complements the empirical tactic matrix from 1A — same numbers, different angle (conditional on tactic *forced* vs *organically chosen*).

In [ ]:
FORCED_TACTICS_CSV = os.path.join(OUT_DIR, "forced_tactics.csv")

if os.path.exists(FORCED_TACTICS_CSV):
    print(f"SKIP — forced tactics CSV already exists.")
else:
    # Stratified sample: 1-3 per (retinue, MPC) bucket
    by_bucket = defaultdict(list)
    for ld in pool:
        by_bucket[(ld.retinue, ld.military_pursuit_count)].append(ld)
    sample = []
    rng = random.Random(2026)
    for key, lds in by_bucket.items():
        sample.extend(rng.sample(lds, min(len(lds), 2)))
    if len(sample) > FORCED_SAMPLE_SIZE:
        sample = rng.sample(sample, FORCED_SAMPLE_SIZE)
    print(f"Forced-tactics sample: {len(sample)} loadouts")

    t0 = time.time()
    result = tactics_analysis.empirical_tactic_matrix(
        sample, n_runs=N_RUNS_FORCED, verbose=False, n_workers=N_WORKERS_MAIN)
    print(f"Forced-tactics sweep complete in {time.time()-t0:.0f}s")

    # Flatten into a long-form CSV
    rows = []
    for i, a_t in enumerate(TACTICS):
        for j, b_t in enumerate(TACTICS):
            rows.append({
                "a_tactic": a_t, "b_tactic": b_t,
                "win_rate": result["win_rate"].iloc[i, j],
                "survival": result["survival"].iloc[i, j],
                "skirm": result["skirm"].iloc[i, j],
                "indecisive_rate": result["indecisive"].iloc[i, j],
            })
    pd.DataFrame(rows).to_csv(FORCED_TACTICS_CSV, index=False)
    print(f"Wrote {FORCED_TACTICS_CSV}")

## 1D. Horde mode (multi-battle survival)

Each loadout fights `HORDE_BATTLES` consecutive battles with carry-over fatigue, Strain accumulation, and Apothecary heal between battles. Measures sustained performance, not just one-battle wins.

In [ ]:
HORDE_CSV = os.path.join(OUT_DIR, "horde_survival.csv")

if os.path.exists(HORDE_CSV):
    print(f"SKIP — horde survival CSV already exists.")
else:
    try:
        import horde_mode
        importlib.reload(horde_mode)
    except ImportError:
        print("horde_mode.py not found — skipping. (Section 7 will be empty.)")
    else:
        rng = random.Random(2026)
        by_bucket = defaultdict(list)
        for ld in pool:
            by_bucket[(ld.retinue, ld.military_pursuit_count)].append(ld)
        sample = []
        for key, lds in by_bucket.items():
            sample.extend(rng.sample(lds, min(len(lds), 2)))
        if len(sample) > HORDE_SAMPLE_SIZE:
            sample = rng.sample(sample, HORDE_SAMPLE_SIZE)

        print(f"Horde sample: {len(sample)} loadouts × {HORDE_BATTLES} consecutive battles × {N_RUNS_HORDE} runs")
        t0 = time.time()
        rows = []
        for i, ld in enumerate(sample):
            survival = horde_mode.run_horde(ld, sample, n_battles=HORDE_BATTLES, n_runs=N_RUNS_HORDE, seed=2026+i*1009)
            rows.append({
                "name": ld.name, "retinue": ld.retinue,
                "mpc": ld.military_pursuit_count, "domain_count": ld.domain_count,
                "battles_survived_mean": survival.get("battles_survived_mean", 0),
                "battles_survived_median": survival.get("battles_survived_median", 0),
                "size_remaining_mean": survival.get("final_size_mean", 0),
            })
        pd.DataFrame(rows).to_csv(HORDE_CSV, index=False)
        print(f"Horde mode complete in {time.time()-t0:.0f}s — wrote {HORDE_CSV}")

## 1E. Load all CSVs into DataFrames for analysis

All sections below read from these dataframes — no further computation needed.

In [ ]:
# Main tournament summary (per-loadout aggregates) — load raw
df_main_summary = pd.read_csv(MAIN_SUMMARY)

# Main tournament matchups — must go through analysis.load_tournament() to get derived columns
# (n_runs, a_win_rate, b_win_rate, a_killed, b_killed, a_upkeep, b_upkeep, spoils, etc.)
# Bare pd.read_csv() will produce KeyError downstream in every analysis function.
df_main_matchups = analysis.load_tournament(MAIN_MATCHUPS) if os.path.exists(MAIN_MATCHUPS) else None
df_tactic_matrix = pd.read_csv(MAIN_TACTIC_MATRIX) if os.path.exists(MAIN_TACTIC_MATRIX) else None

# Playstyle tournament
df_ps_summary = pd.read_csv(PLAYSTYLE_SUMMARY) if os.path.exists(PLAYSTYLE_SUMMARY) else None
df_ps_matchups = analysis.load_tournament(PLAYSTYLE_MATCHUPS) if os.path.exists(PLAYSTYLE_MATCHUPS) else None

# Forced tactics
df_forced_tactics = pd.read_csv(FORCED_TACTICS_CSV) if os.path.exists(FORCED_TACTICS_CSV) else None

# Horde
df_horde = pd.read_csv(HORDE_CSV) if os.path.exists(HORDE_CSV) else None

print(f"Main summary: {df_main_summary.shape}")
print(f"Main matchups: {df_main_matchups.shape if df_main_matchups is not None else 'missing'}")
print(f"Tactic matrix: {df_tactic_matrix.shape if df_tactic_matrix is not None else 'missing'}")
print(f"Playstyle summary: {df_ps_summary.shape if df_ps_summary is not None else 'missing'}")
print(f"Playstyle matchups: {df_ps_matchups.shape if df_ps_matchups is not None else 'missing'}")
print(f"Forced tactics: {df_forced_tactics.shape if df_forced_tactics is not None else 'missing'}")
print(f"Horde: {df_horde.shape if df_horde is not None else 'missing'}")

# 2. Pursuit & Domain Analysis

**Design target:** win rate should be tightly correlated to Military Pursuit Count (MPC). A player spending 13pts on military should beat a player spending 5pts. This section evaluates how cleanly that correlation holds, and whether Domain Count (the breadth-of-investment metric) tells the same story or a different one.

## 2.1 Correlation: Win Rate ↔ MPC

The headline number. Pearson and Spearman correlations of per-loadout win rate vs MPC.

In [ ]:
from scipy import stats as sp_stats

g = df_main_summary
pearson_r, pearson_p = sp_stats.pearsonr(g["military_pursuit_count"], g["win_rate"])
spearman_r, spearman_p = sp_stats.spearmanr(g["military_pursuit_count"], g["win_rate"])

print(f"=== Win Rate ↔ Military Pursuit Count ===")
print(f"  Pearson:  r = {pearson_r:+.3f}  (p = {pearson_p:.2e})")
print(f"  Spearman: r = {spearman_r:+.3f}  (p = {spearman_p:.2e})")
print()
print(f"Interpretation:")
print(f"  +1.0  perfect linear correlation: MPC fully predicts win rate.")
print(f"  +0.7+ strong: design target met.")
print(f"  +0.4-0.7 moderate: budget matters but other factors strong.")
print(f"  <+0.4 weak: budget barely matters — investigate why.")

In [ ]:
# Visualize: MPC vs Win Rate scatter + binned mean line
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(g["military_pursuit_count"], g["win_rate"], alpha=0.15, s=20, color="steelblue")
binned = g.groupby("military_pursuit_count")["win_rate"].agg(["mean", "std", "count"])
ax.errorbar(binned.index, binned["mean"], yerr=binned["std"], fmt="o-", color="darkred",
            markersize=10, linewidth=2, capsize=4, label="Mean ± std per MPC")
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5, label="50% win rate")
ax.set_xlabel("Military Pursuit Count (MPC)")
ax.set_ylabel("Win Rate")
ax.set_title(f"Win Rate vs Military Pursuit Count  (Pearson r = {pearson_r:+.3f})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2.2 MPC bucket summary — does higher always win?

Test: for every loadout at MPC=N, what fraction of opponents at MPC<N do they beat? At MPC>N? Cross-budget head-to-head is the strict test of "spending more should win more."

In [ ]:
# Build cross-MPC win rate matrix from matchups
df_m = df_main_matchups
df_m["a_mpc_bucket"] = df_m["a_military_pursuit_count"]
df_m["b_mpc_bucket"] = df_m["b_military_pursuit_count"]
# Aggregate wins/losses per (a_mpc, b_mpc) pair
grp = df_m.groupby(["a_mpc_bucket", "b_mpc_bucket"]).agg(
    a_wins=("a_wins", "sum"), b_wins=("b_wins", "sum"),
    mut=("mut_wipe", "sum"), indec=("indecisive", "sum"),
).reset_index()
grp["total"] = grp[["a_wins","b_wins","mut","indec"]].sum(axis=1)
grp["a_win_rate"] = grp["a_wins"] / grp["total"]
grp["decisive_rate"] = (grp["a_wins"] + grp["b_wins"]) / grp["total"]

# Pivot: rows = A's MPC, cols = B's MPC, value = A's win rate
pivot_wr = grp.pivot(index="a_mpc_bucket", columns="b_mpc_bucket", values="a_win_rate")
pivot_dec = grp.pivot(index="a_mpc_bucket", columns="b_mpc_bucket", values="decisive_rate")

print("=== A's Win Rate by (A_MPC, B_MPC) — values show how often A beats B ===")
print(pivot_wr.round(3).to_string())
print()
print("Diagonal = mirror MPC. Above diagonal: A has more budget than B (should win).")

In [ ]:
# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
im = ax.imshow(pivot_wr.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot_wr.columns))); ax.set_xticklabels(pivot_wr.columns)
ax.set_yticks(range(len(pivot_wr.index))); ax.set_yticklabels(pivot_wr.index)
ax.set_xlabel("Opponent (B) MPC"); ax.set_ylabel("Player (A) MPC")
ax.set_title("A's Win Rate by MPC matchup")
plt.colorbar(im, ax=ax)
# Annotate
for i in range(len(pivot_wr.index)):
    for j in range(len(pivot_wr.columns)):
        v = pivot_wr.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="black", fontsize=8)

ax = axes[1]
# Average win rate over LOWER vs HIGHER MPC opponents per row
rows = []
for a_mpc in sorted(pivot_wr.index):
    higher = pivot_wr.loc[a_mpc, pivot_wr.columns > a_mpc].dropna()
    lower  = pivot_wr.loc[a_mpc, pivot_wr.columns < a_mpc].dropna()
    same   = pivot_wr.loc[a_mpc, [a_mpc] if a_mpc in pivot_wr.columns else []]
    rows.append({
        "mpc": a_mpc,
        "vs_lower_mean": lower.mean() if len(lower) else np.nan,
        "vs_higher_mean": higher.mean() if len(higher) else np.nan,
        "vs_same_mean": same.mean() if len(same) else np.nan,
    })
df_vs = pd.DataFrame(rows)
ax.plot(df_vs["mpc"], df_vs["vs_lower_mean"], "o-", color="green", label="vs lower MPC")
ax.plot(df_vs["mpc"], df_vs["vs_same_mean"], "s--", color="gray", label="vs same MPC")
ax.plot(df_vs["mpc"], df_vs["vs_higher_mean"], "^-", color="red", label="vs higher MPC")
ax.axhline(0.5, color="black", linestyle=":", alpha=0.5)
ax.set_xlabel("Player's MPC"); ax.set_ylabel("Win rate")
ax.set_title("Performance by opponent's budget bracket")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nKey check: does the GREEN line stay above 0.5 (you beat lower-MPC opponents)?")
print("Does the RED line stay below 0.5 (you lose to higher-MPC opponents)?")

## 2.3 Can the 30-DomainCount player lose to the 16-DomainCount player?

Same question for Domain Count: does broader domain investment correlate with winning, and can a heavily-invested player be beaten by a lean one?

In [ ]:
pearson_d, _ = sp_stats.pearsonr(g["domain_count"], g["win_rate"])
spearman_d, _ = sp_stats.spearmanr(g["domain_count"], g["win_rate"])
print(f"=== Win Rate ↔ Domain Count ===")
print(f"  Pearson:  r = {pearson_d:+.3f}")
print(f"  Spearman: r = {spearman_d:+.3f}")

# MPC↔DC are highly correlated by construction. Partial: residual win rate AFTER removing MPC effect.
from sklearn.linear_model import LinearRegression
X_mpc = g[["military_pursuit_count"]].values
y = g["win_rate"].values
mpc_model = LinearRegression().fit(X_mpc, y)
y_residual = y - mpc_model.predict(X_mpc)
# Now correlate residual with DC
res_pearson, _ = sp_stats.pearsonr(g["domain_count"], y_residual)
print(f"\nWin Rate residual (after MPC removed) ↔ Domain Count:")
print(f"  Pearson:  r = {res_pearson:+.3f}")
print(f"  → If close to 0, DC doesn't add independent predictive value beyond MPC.")
print(f"  → If positive, broader domain investment HELPS independently of military spend.")
print(f"  → If negative, broader investment HURTS (suggests pursuit redundancy / wasted breadth).")

In [ ]:
# Specific test: how often does a high-DC loader lose to a low-DC loader?
# Build cross-DC matchup table
df_m["a_dc_bucket"] = (df_m["a_domain_count"] // 5) * 5  # 5-pt buckets
df_m["b_dc_bucket"] = (df_m["b_domain_count"] // 5) * 5
grp_dc = df_m.groupby(["a_dc_bucket", "b_dc_bucket"]).agg(
    a_wins=("a_wins","sum"), b_wins=("b_wins","sum"),
    mut=("mut_wipe","sum"), indec=("indecisive","sum")).reset_index()
grp_dc["total"] = grp_dc[["a_wins","b_wins","mut","indec"]].sum(axis=1)
grp_dc["a_win_rate"] = grp_dc["a_wins"] / grp_dc["total"]
pivot_dc = grp_dc.pivot(index="a_dc_bucket", columns="b_dc_bucket", values="a_win_rate")

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(pivot_dc.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot_dc.columns))); ax.set_xticklabels(pivot_dc.columns)
ax.set_yticks(range(len(pivot_dc.index))); ax.set_yticklabels(pivot_dc.index)
ax.set_xlabel("Opponent (B) Domain Count bucket")
ax.set_ylabel("Player (A) Domain Count bucket")
ax.set_title("A's Win Rate by Domain Count matchup")
plt.colorbar(im, ax=ax)
for i in range(len(pivot_dc.index)):
    for j in range(len(pivot_dc.columns)):
        v = pivot_dc.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="black", fontsize=8)
plt.tight_layout(); plt.show()

## 2.4 Outliers — punch above weight on MPC

Loadouts whose win rate is most above what their MPC bucket would predict — "value loadouts." Conversely, loadouts that under-perform their budget.

In [ ]:
over, under = analysis.mpc_outliers(df_main_matchups, top_n=15)
print("=== Top punch-above-weight loadouts (most over-perform their MPC peers) ===")
print(over[["retinue","military_pursuit_count","win_rate","bucket_mean_wr","z_score"]].round(3).to_string())
print("\n=== Top under-performers (waste budget) ===")
print(under[["retinue","military_pursuit_count","win_rate","bucket_mean_wr","z_score"]].round(3).to_string())

# 3. Retinue Analysis

How does win rate, durability, kill efficiency, and stalemate rate vary across the four retinues? Are retinue cost-efficiency ratios appropriate?

In [ ]:
print("=== Per-Retinue summary (all loadouts, Random playstyle) ===")
ret_summary = df_main_summary.groupby("retinue").agg(
    n_loadouts=("name", "count"),
    win_rate=("win_rate", "mean"),
    loss_rate=("loss_rate", "mean"),
    decisive_win_rate=("decisive_win_rate", "mean"),
    kill_eff=("kill_efficiency", "mean"),
    survivors=("avg_self_survivors", "mean"),
    opp_survivors=("avg_opp_survivors", "mean"),
    wins_per_1k_upkeep=("wins_per_1000_upkeep", "mean"),
    upkeep=("upkeep_per_retinue", "mean"),
)
print(ret_summary.round(3).to_string())

In [ ]:
# Win rate by retinue, faceted by MPC
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for ret, color in zip(["Levy","Man-at-Arms","Sergeant","Knight Templar"],
                       ["#888","#4a90e2","#d4a800","#c43838"]):
    sub = df_main_summary[df_main_summary["retinue"] == ret]
    binned = sub.groupby("military_pursuit_count")["win_rate"].agg(["mean","std","count"])
    binned = binned[binned["count"] >= 3]
    if len(binned) > 0:
        ax.errorbar(binned.index, binned["mean"], yerr=binned["std"],
                    fmt="o-", color=color, label=ret, capsize=3, markersize=8)
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Military Pursuit Count")
ax.set_ylabel("Win Rate")
ax.set_title("Win Rate by MPC × Retinue")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ret_means = ret_summary["win_rate"].sort_values(ascending=False)
colors = {"Levy":"#888","Man-at-Arms":"#4a90e2","Sergeant":"#d4a800","Knight Templar":"#c43838"}
ax.bar(ret_means.index, ret_means.values, color=[colors[r] for r in ret_means.index])
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_ylabel("Mean Win Rate")
ax.set_title("Mean Win Rate by Retinue")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

## 3.1 Retinue cost efficiency

Wins per 1000 gold upkeep — does spending more on better troops pay off?

In [ ]:
ret_costs = df_main_summary.groupby("retinue").agg(
    army_upkeep=("army_upkeep", "first"),
    wins_per_1k=("wins_per_1000_upkeep", "mean"),
    win_rate=("win_rate", "mean"),
).round(2)
ret_costs["upkeep_50_men"] = ret_costs["army_upkeep"]
print("=== Retinue cost-efficiency ===")
print(ret_costs.to_string())

# 4. Equipment Analysis

Weapons, shields, armor, and 1H/2H/Bastard tradeoffs.

## 4.1 Weapon performance

In [ ]:
weap = df_main_summary.groupby("weapon").agg(
    n_loadouts=("name","count"),
    win_rate=("win_rate","mean"),
    decisive_win_rate=("decisive_win_rate","mean"),
    kill_eff=("kill_efficiency","mean"),
).sort_values("win_rate", ascending=False)
weap = weap[weap["n_loadouts"] >= 5]
print("=== Weapon performance (≥5 loadouts) ===")
print(weap.round(3).to_string())

## 4.2 Shield performance

In [ ]:
sh = df_main_summary.copy()
sh["shield"] = sh["shield"].fillna("None")
sh = sh.groupby("shield").agg(
    n=("name","count"),
    win_rate=("win_rate","mean"),
    decisive_win_rate=("decisive_win_rate","mean"),
    kill_eff=("kill_efficiency","mean"),
).sort_values("win_rate", ascending=False)
print("=== Shield performance ===")
print(sh.round(3).to_string())

## 4.3 Armor performance

In [ ]:
arm = df_main_summary.groupby("armor").agg(
    n=("name","count"),
    win_rate=("win_rate","mean"),
    decisive_win_rate=("decisive_win_rate","mean"),
    survivors=("avg_self_survivors","mean"),
).sort_values("win_rate", ascending=False)
print("=== Armor performance ===")
print(arm.round(3).to_string())

## 4.4 Bastard Sword — does the dual-profile actually help?

Compare Bastard 1H+shield builds vs Bastard 2H builds vs other 2H weapons (Poleaxe, Halberd, etc.).

In [ ]:
bast = df_main_summary[df_main_summary["weapon"] == "Bastard Sword"].copy()
bast["mode"] = np.where(bast["shield"].fillna("None") == "None", "Bastard 2H", "Bastard 1H+shield")
b_summary = bast.groupby("mode").agg(
    n=("name","count"),
    win_rate=("win_rate","mean"),
    decisive_win_rate=("decisive_win_rate","mean"),
    kill_eff=("kill_efficiency","mean"),
).round(3)
print("=== Bastard Sword variants ===")
print(b_summary.to_string())

# Compare to other heavy weapons
print()
print("=== Other heavy 2H weapons for reference ===")
other = df_main_summary[df_main_summary["weapon"].isin(["Poleaxe","Halberd","War Hammer","Battle Axe"])]
print(other.groupby("weapon").agg(
    n=("name","count"),
    win_rate=("win_rate","mean"),
    decisive_win_rate=("decisive_win_rate","mean"),
).round(3).to_string())

## 4.5 Lance — does Charge synergy pay off?

In [ ]:
lance = df_main_summary[df_main_summary["weapon"] == "Lance"]
non_lance = df_main_summary[df_main_summary["weapon"] != "Lance"]
print(f"Lance loadouts:  n={len(lance):3d}  win_rate={lance['win_rate'].mean():.3f}  decisive_wr={lance['decisive_win_rate'].mean():.3f}")
print(f"Non-Lance:       n={len(non_lance):3d}  win_rate={non_lance['win_rate'].mean():.3f}  decisive_wr={non_lance['decisive_win_rate'].mean():.3f}")
# Best lance shields
if len(lance) > 0:
    print("\nLance shield breakdown:")
    print(lance.groupby("shield").agg(n=("name","count"), wr=("win_rate","mean")).round(3).to_string())

## 4.6 Shield deep-dive

A focused analysis of shield builds across all metrics. Shield types map to gear tiers (Wooden=Crude, Kite=Cast, Scutum=Wrought, Tower=Forged, Heater=Crafted) per the loadout rules — so this also doubles as a tier-by-tier comparison of shielded builds.

### 4.6.1 Win rate by shield type

In [ ]:
# Filter to shield-bearing matchup rows (A side)
shield_df = df_main_matchups[df_main_matchups['a_shield'].notna() & (df_main_matchups['a_shield'] != '')].copy()
shield_df['n_battles'] = shield_df['a_wins'] + shield_df['b_wins'] + shield_df['mut_wipe'] + shield_df['indecisive']
print(f'Shield-bearing matchup rows: {len(shield_df):,} (of {len(df_main_matchups):,} total)')

# Per-loadout aggregation
shield_winrate = shield_df.groupby(['a_name', 'a_shield', 'a_retinue', 'a_weapon', 'a_armor']).agg(
    n_battles=('n_battles', 'sum'),
    wins=('a_wins', 'sum'),
    losses=('b_wins', 'sum'),
    mutual=('mut_wipe', 'sum'),
).reset_index()
shield_winrate['win_rate'] = shield_winrate['wins'] / shield_winrate['n_battles']
shield_winrate['decisive_win_rate'] = shield_winrate['wins'] / shield_winrate[['wins','losses']].sum(axis=1).clip(lower=1)
shield_winrate = shield_winrate.sort_values('win_rate', ascending=False)

print('\n=== Win rate by shield type (aggregate) ===')
by_type = shield_winrate.groupby('a_shield').agg(
    n_loadouts=('a_name', 'count'),
    avg_win_rate=('win_rate', 'mean'),
    avg_decisive_wr=('decisive_win_rate', 'mean'),
).sort_values('avg_win_rate', ascending=False)
print(by_type.round(3).to_string())

### 4.6.2 Durability — survival rates by shield

In [ ]:
# Use analysis.durability and join shield info back in
dur = analysis.durability(df_main_matchups)
shield_lookup = df_main_matchups.groupby('a_name')['a_shield'].first().to_dict()
dur['shield'] = dur['a_name'].map(shield_lookup).fillna('None')

dur_by_shield = dur.groupby('shield').agg(
    n_loadouts=('a_name', 'count'),
    avg_survival_rate=('avg_survival_rate', 'mean'),
).sort_values('avg_survival_rate', ascending=False)
print('=== Survival rate by shield type ===')
print(dur_by_shield.round(3).to_string())

print('\n=== Top 10 most durable shield builds ===')
top_shield_dur = dur[dur['shield'] != 'None'].nlargest(10, 'avg_survival_rate')[
    ['a_name','shield','avg_survival_rate']]
print(top_shield_dur.round(3).to_string(index=False))

### 4.6.3 Shield destruction rate

How often does each shield get smashed mid-battle? Wooden + Cast tier shields are most fragile; Heater the toughest.

In [ ]:
# Drop mirror matchups
sd_df = shield_df[shield_df['a_name'] != shield_df['b_name']].copy()
destroy_by_type = sd_df.groupby('a_shield').agg(
    n_matchups=('a_shield', 'count'),
    avg_destroy_pct=('a_shield_destroyed_rate', lambda x: x.mean() * 100),
).sort_values('avg_destroy_pct', ascending=False)
print('=== Avg shield destruction rate by type ===')
print(destroy_by_type.round(1).to_string())

# Top 10 most-destroyed shield builds
print('\n=== Top 10 most-destroyed shield builds ===')
top_destroyed = sd_df.groupby(['a_name', 'a_shield']).agg(
    avg_destroy_pct=('a_shield_destroyed_rate', lambda x: x.mean() * 100),
).reset_index().sort_values('avg_destroy_pct', ascending=False).head(10)
print(top_destroyed.round(1).to_string(index=False))

### 4.6.4 What counters shield builds?

For each shield type, which weapons/loadouts beat them most reliably?

In [ ]:
def top_counters_for_shield(shield_name, top_n=8):
    rows = df_main_matchups[
        (df_main_matchups['a_shield'] == shield_name) &
        (df_main_matchups['a_name'] != df_main_matchups['b_name'])
    ].copy()
    if len(rows) == 0:
        return None
    rows['n_battles'] = rows['a_wins'] + rows['b_wins'] + rows['mut_wipe'] + rows['indecisive']
    rows['a_winrate'] = rows['a_wins'] / rows['n_battles']
    rows['b_winrate'] = rows['b_wins'] / rows['n_battles']
    counters = rows.groupby(['b_weapon', 'b_retinue', 'b_shield']).agg(
        b_wr=('b_winrate', 'mean'),
        a_wr=('a_winrate', 'mean'),
        n=('a_name', 'count'),
    ).reset_index()
    counters['margin'] = counters['b_wr'] - counters['a_wr']
    counters = counters[counters['n'] >= 5].sort_values('margin', ascending=False).head(top_n)
    return counters

for shield_type in ['Wooden Shield','Kite Shield','Scutum Shield','Tower Shield','Heater Shield']:
    c = top_counters_for_shield(shield_type)
    if c is not None and len(c) > 0:
        print(f'\n=== Top counters for {shield_type} ===')
        print(c.round(3).to_string(index=False))

### 4.6.5 What do shield builds dominate?

Inverse — for each shield type, what loadouts do they beat reliably?

In [ ]:
def top_targets_for_shield(shield_name, top_n=8):
    rows = df_main_matchups[
        (df_main_matchups['a_shield'] == shield_name) &
        (df_main_matchups['a_name'] != df_main_matchups['b_name'])
    ].copy()
    if len(rows) == 0:
        return None
    rows['n_battles'] = rows['a_wins'] + rows['b_wins'] + rows['mut_wipe'] + rows['indecisive']
    rows['a_winrate'] = rows['a_wins'] / rows['n_battles']
    rows['b_winrate'] = rows['b_wins'] / rows['n_battles']
    targets = rows.groupby(['b_weapon', 'b_retinue', 'b_shield']).agg(
        a_wr=('a_winrate', 'mean'),
        b_wr=('b_winrate', 'mean'),
        n=('a_name', 'count'),
    ).reset_index()
    targets['margin'] = targets['a_wr'] - targets['b_wr']
    targets = targets[targets['n'] >= 5].sort_values('margin', ascending=False).head(top_n)
    return targets

for shield_type in ['Wooden Shield','Kite Shield','Scutum Shield','Tower Shield','Heater Shield']:
    t = top_targets_for_shield(shield_type)
    if t is not None and len(t) > 0:
        print(f'\n=== Top targets (favorable matchups) for {shield_type} ===')
        print(t.round(3).to_string(index=False))

### 4.6.6 Best tactics for shield builds

For one representative loadout per shield type, what's the best opening tactic?

In [ ]:
# Pick the top-winrate loadout for each shield type from shield_winrate
representatives = []
for shield_type in ['Wooden Shield', 'Kite Shield', 'Scutum Shield', 'Tower Shield', 'Heater Shield']:
    matches = shield_winrate[shield_winrate['a_shield'] == shield_type]
    if len(matches) > 0:
        representatives.append(matches.iloc[0]['a_name'])

# Use the empirical tactic matrix (loaded in Section 1E) as a proxy for tactic guidance.
# For per-loadout tactic profile, see Section 5.
if df_tactic_matrix is not None:
    _tac_pivot = df_tactic_matrix.pivot(index='a_tactic', columns='b_tactic', values='a_win_rate')
    _tac_pivot = _tac_pivot.reindex(index=TACTICS, columns=TACTICS)
    marginal_wr = _tac_pivot.mean(axis=1).sort_values(ascending=False)

    print('Marginal tactic win rates (best openings against a uniform-tactic opponent):')
    for t, wr in marginal_wr.items():
        print(f'  {t:<22} {wr:.3f}')

    print(f'\nRepresentative shield builds (best win-rate loadout per shield type):')
    for name in representatives:
        print(f'  {name}')
else:
    print('Tactic matrix not loaded — skip')

### 4.6.7 Shield summary

In [ ]:
print('=== Shield Findings Summary ===')
print()
print('1. Win rate by shield (mean across loadouts):')
print(by_type[['n_loadouts','avg_win_rate','avg_decisive_wr']].round(3).to_string())
print()
print('2. Durability ranking:')
print(dur_by_shield.round(3).to_string())
print()
print('3. Destruction frequency (lower = tougher shield):')
print(destroy_by_type.round(1).to_string())

## 4.7 Bastard Sword deep-dive

The dual-profile weapon: 1H mode (with shield) provides Shatter Armor + Steady; 2H mode (no shield, OR shield destroyed) provides Cleave + Unwieldy. Adaptive: a Bastard+shield build automatically switches to 2H mode if the shield breaks. How does this adaptive mechanic perform in practice?

### 4.7.1 Overall comparison — Bastard 1H vs 2H vs other

In [ ]:
bastard_df = df_main_matchups[df_main_matchups['a_weapon'] == 'Bastard Sword'].copy()
bastard_df['n_battles'] = bastard_df['a_wins'] + bastard_df['b_wins'] + bastard_df['mut_wipe'] + bastard_df['indecisive']
bastard_df['mode'] = bastard_df['a_shield'].apply(
    lambda s: '2H (no shield)' if (pd.isna(s) or s == '') else f'1H + {s}')
print(f'Bastard Sword matchup rows: {len(bastard_df):,}\n')

mode_winrate = bastard_df.groupby(['a_name', 'mode', 'a_retinue', 'a_armor']).agg(
    n_battles=('n_battles', 'sum'),
    wins=('a_wins', 'sum'),
    losses=('b_wins', 'sum'),
).reset_index()
mode_winrate['win_rate'] = mode_winrate['wins'] / mode_winrate['n_battles']
mode_winrate['decisive_wr'] = mode_winrate['wins'] / mode_winrate[['wins','losses']].sum(axis=1).clip(lower=1)
mode_winrate = mode_winrate.sort_values('win_rate', ascending=False)

print('=== Bastard variants — mean win rate by mode ===')
by_mode = mode_winrate.groupby('mode').agg(
    n_loadouts=('a_name', 'count'),
    avg_win_rate=('win_rate', 'mean'),
    avg_decisive_wr=('decisive_wr', 'mean'),
).sort_values('avg_win_rate', ascending=False)
print(by_mode.round(3).to_string())
print('\n=== Top 10 Bastard builds (any mode) ===')
print(mode_winrate.head(10)[['a_name','mode','a_retinue','win_rate','decisive_wr']].round(3).to_string(index=False))

### 4.7.2 Adaptive value — does mode-switching pay off?

How often does a Bastard+shield build have its shield destroyed mid-battle, forcing the 2H mode switch? Pair that with win rate to see whether the adaptive mechanic is actually delivering value.

In [ ]:
adaptive_df = bastard_df[(bastard_df['a_shield'].notna()) & (bastard_df['a_shield'] != '')].copy()
adaptive_df = adaptive_df[adaptive_df['a_name'] != adaptive_df['b_name']]
destruction = adaptive_df.groupby(['a_name', 'a_shield']).agg(
    avg_destroy_pct=('a_shield_destroyed_rate', lambda x: x.mean() * 100),
    n_opponents=('a_name', 'count'),
).reset_index().sort_values('avg_destroy_pct', ascending=False)

print('=== Top 15 Bastard 1H builds by shield destruction rate ===')
print('(High % = mode-switch triggered often — adaptive mechanic in active use)\n')
print(destruction.head(15).round(1).to_string(index=False))

# Compare: does high destruction correlate with better or worse win rate?
merged = mode_winrate[mode_winrate['mode'].str.contains('1H')].merge(
    destruction, on='a_name', how='inner')
if len(merged) > 0:
    from scipy import stats as sp_stats
    r, p = sp_stats.pearsonr(merged['avg_destroy_pct'], merged['win_rate'])
    print(f'\nDestruction rate ↔ Bastard 1H win rate: Pearson r = {r:+.3f}')
    print('  Positive: more destruction = better outcomes (2H mode rewarding)')
    print('  Negative: destruction is a death spiral')

### 4.7.3 Counters for each Bastard mode

In [ ]:
def bastard_counters(filter_func, label, n=8):
    rows = df_main_matchups[filter_func(df_main_matchups) &
                            (df_main_matchups['a_name'] != df_main_matchups['b_name'])].copy()
    if len(rows) == 0:
        return
    rows['n_battles'] = rows['a_wins'] + rows['b_wins'] + rows['mut_wipe'] + rows['indecisive']
    rows['a_wr'] = rows['a_wins'] / rows['n_battles']
    rows['b_wr'] = rows['b_wins'] / rows['n_battles']
    counters = rows.groupby(['b_weapon', 'b_retinue', 'b_shield']).agg(
        b_wr=('b_wr', 'mean'),
        a_wr=('a_wr', 'mean'),
        n_pairs=('a_name', 'count'),
    ).reset_index()
    counters['margin'] = counters['b_wr'] - counters['a_wr']
    counters = counters[counters['n_pairs'] >= 5].sort_values('margin', ascending=False).head(n)
    print(f'\n=== Top counters for {label} ===')
    print(counters.round(3).to_string(index=False))

# 1H mode (has shield)
bastard_counters(
    lambda d: (d['a_weapon'] == 'Bastard Sword') & d['a_shield'].notna() & (d['a_shield'] != ''),
    'Bastard 1H+shield')
# 2H mode (no shield)
bastard_counters(
    lambda d: (d['a_weapon'] == 'Bastard Sword') & ((d['a_shield'].isna()) | (d['a_shield'] == '')),
    'Bastard 2H')

### 4.7.4 Best shield pairing for Bastard 1H

In [ ]:
bastard_1h = bastard_df[(bastard_df['a_shield'].notna()) & (bastard_df['a_shield'] != '')].copy()
pairing = bastard_1h.groupby(['a_shield', 'a_retinue', 'a_armor']).agg(
    n_loadouts=('a_name', 'nunique'),
    n_battles=('n_battles', 'sum'),
    wins=('a_wins', 'sum'),
    avg_destroy_pct=('a_shield_destroyed_rate', lambda x: x.mean() * 100),
).reset_index()
pairing['win_rate'] = pairing['wins'] / pairing['n_battles']
pairing = pairing.sort_values('win_rate', ascending=False)
print('=== Bastard 1H × shield × armor combos (top 15 by win rate) ===')
print(pairing.head(15).round(3).to_string(index=False))

### 4.7.5 Bastard vs other top weapons

In [ ]:
focus_weapons = ['Bastard Sword', 'Poleaxe', 'Lance', 'Halberd', 'Pike', 'Battle Axe', 'Morningstar', 'War Hammer']
focus = df_main_matchups[
    df_main_matchups['a_weapon'].isin(focus_weapons) &
    (df_main_matchups['a_name'] != df_main_matchups['b_name'])
].copy()
focus['n_battles'] = focus['a_wins'] + focus['b_wins'] + focus['mut_wipe'] + focus['indecisive']

def to_mode(row):
    if row['a_weapon'] == 'Bastard Sword':
        return 'Bastard 1H' if pd.notna(row['a_shield']) and row['a_shield'] != '' else 'Bastard 2H'
    return row['a_weapon']
focus['mode'] = focus.apply(to_mode, axis=1)

comparison = focus.groupby(['mode', 'a_retinue']).agg(
    n_loadouts=('a_name', 'nunique'),
    n_battles=('n_battles', 'sum'),
    wins=('a_wins', 'sum'),
    losses=('b_wins', 'sum'),
).reset_index()
comparison['win_rate'] = comparison['wins'] / comparison['n_battles']
comparison['decisive_wr'] = comparison['wins'] / comparison[['wins','losses']].sum(axis=1).clip(lower=1)
comparison = comparison.sort_values(['a_retinue', 'win_rate'], ascending=[True, False])
print('=== Heavy weapons by retinue ===')
print(comparison[['a_retinue','mode','n_loadouts','win_rate','decisive_wr']].round(3).to_string(index=False))

### 4.7.6 Bastard summary

In [ ]:
print('=== Bastard Sword findings ===\n')
print('1. Mode comparison:')
print(by_mode.round(3).to_string())
print()
top_destroy = destruction['avg_destroy_pct'].mean() if len(destruction) > 0 else 0
print(f'2. Average shield destruction rate (Bastard 1H builds): {top_destroy:.1f}%')
print(f'   (Higher = adaptive switch fires often)')
print()
print('3. Best Bastard 1H combo (by win rate):')
print(pairing.head(3)[['a_shield','a_retinue','a_armor','win_rate','avg_destroy_pct']].round(3).to_string(index=False))

# 5. Tactics Analysis

Two complementary views:
1. **Empirical matrix** from the main tournament — what *actually happens* when players choose tactics under Random policy
2. **Forced matrix** — head-to-head under deterministic tactic forcing

## 5.1 Empirical tactic matrix (from main tournament)

These are the win rates when A and B chose their first-skirmish tactics organically (per Random policy) and the battle played out. No tactics were forced.

In [ ]:
pivot_emp = df_tactic_matrix.pivot(index="a_tactic", columns="b_tactic", values="a_win_rate")
pivot_emp = pivot_emp.reindex(index=TACTICS, columns=TACTICS)
pivot_indec = df_tactic_matrix.pivot(index="a_tactic", columns="b_tactic", values="stalemate_rate")
pivot_indec = pivot_indec.reindex(index=TACTICS, columns=TACTICS)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
im = ax.imshow(pivot_emp.values, cmap="RdYlGn", vmin=0, vmax=0.5, aspect="auto")
ax.set_xticks(range(7)); ax.set_xticklabels(TACTICS, rotation=45, ha="right")
ax.set_yticks(range(7)); ax.set_yticklabels(TACTICS)
ax.set_title("Empirical: A's win rate by (A_first_tactic, B_first_tactic)")
ax.set_xlabel("B opens with"); ax.set_ylabel("A opens with")
plt.colorbar(im, ax=ax)
for i in range(7):
    for j in range(7):
        v = pivot_emp.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)

ax = axes[1]
im = ax.imshow(pivot_indec.values, cmap="Reds", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(7)); ax.set_xticklabels(TACTICS, rotation=45, ha="right")
ax.set_yticks(range(7)); ax.set_yticklabels(TACTICS)
ax.set_title("Stalemate rate by tactic opening")
plt.colorbar(im, ax=ax)
for i in range(7):
    for j in range(7):
        v = pivot_indec.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout(); plt.show()

## 5.2 Marginal value of each tactic

Average win rate when opening with each tactic, vs uniform opponent.

In [ ]:
marginal = pivot_emp.mean(axis=1).sort_values(ascending=False)
print("=== Marginal win rate when opening with each tactic ===")
for t, wr in marginal.items():
    print(f"  {t:<22} {wr:.3f}")
print()
print("Read: avg win rate when A opens with this tactic vs a random B opener.")
print("Higher = stronger default opening choice under random play.")

## 5.3 Forced tactics matrix (stratified sample)

For comparison. Every (a, b) tactic pair is forced — pure interaction effect without play-style noise.

In [ ]:
if df_forced_tactics is not None:
    pivot_forced = df_forced_tactics.pivot(index="a_tactic", columns="b_tactic", values="win_rate")
    pivot_forced = pivot_forced.reindex(index=TACTICS, columns=TACTICS)

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(pivot_forced.values, cmap="RdYlGn", vmin=0, vmax=0.5, aspect="auto")
    ax.set_xticks(range(7)); ax.set_xticklabels(TACTICS, rotation=45, ha="right")
    ax.set_yticks(range(7)); ax.set_yticklabels(TACTICS)
    ax.set_title("Forced: A's win rate when A plays row vs B plays column (every skirmish)")
    ax.set_xlabel("B forced to"); ax.set_ylabel("A forced to")
    plt.colorbar(im, ax=ax)
    for i in range(7):
        for j in range(7):
            v = pivot_forced.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

    print("\nDifference (empirical - forced):")
    print((pivot_emp - pivot_forced).round(2).to_string())
else:
    print("Forced tactics CSV not loaded — skip.")

## 5.4 Counter table — which tactic counters which?

For each opponent tactic, which of YOUR tactics produces the best win rate?

In [ ]:
print("=== Best counter for each opponent opening ===")
print(f"{'Opponent plays':<22}  {'Best counter':<22}  {'Win rate':>9}")
for t in TACTICS:
    col = pivot_emp[t].dropna()
    best = col.idxmax(); wr = col.max()
    print(f"  {t:<20}  {best:<22}  {wr:>8.3f}")

# 6. Playstyle Analysis

Compare the playstyle-aware tournament (each loadout uses its assigned best playstyle) against the main Random tournament. Question: do default playstyles actually help loadouts perform?

In [ ]:
if df_ps_summary is None:
    print("Playstyle tournament not loaded — skip.")
else:
    merged = df_main_summary.merge(
        df_ps_summary[["name","win_rate","decisive_win_rate","kill_efficiency","playstyle"]],
        on="name", suffixes=("_random", "_style"))
    merged["wr_delta"] = merged["win_rate_style"] - merged["win_rate_random"]

    print("=== Playstyle vs Random — average delta in win rate ===")
    print(f"  Mean Δ win rate: {merged['wr_delta'].mean():+.4f}")
    print(f"  Median:          {merged['wr_delta'].median():+.4f}")
    print(f"  % helped:        {100*(merged['wr_delta']>0).mean():.1f}%")
    print(f"  % hurt:          {100*(merged['wr_delta']<0).mean():.1f}%")

    print("\n=== By assigned playstyle ===")
    ps_delta = merged.groupby("playstyle").agg(
        n=("name","count"),
        mean_delta=("wr_delta","mean"),
        median_delta=("wr_delta","median"),
        pct_helped=("wr_delta", lambda s: (s > 0).mean()),
    ).round(4)
    print(ps_delta.to_string())

In [ ]:
# Plot
if df_ps_summary is not None:
    fig, ax = plt.subplots(figsize=(11, 6))
    ps_means = merged.groupby("playstyle")["wr_delta"].mean().sort_values()
    colors = ["red" if v < 0 else "green" for v in ps_means.values]
    ax.barh(ps_means.index, ps_means.values, color=colors, alpha=0.7)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Δ win rate (Playstyle - Random)")
    ax.set_title("Default playstyle effect on win rate")
    plt.tight_layout(); plt.show()

## 6.1 KT-Twins Experiment — Unshakable vs Natural Playstyle (within KT)

Each KT loadout was duplicated in the playstyle tournament: one entry uses Unshakable, the other uses the equipment-natural playstyle (Defender, Cavalry, Aggressor, etc). Both have the same KT retinue → same rout-immunity stat. The win-rate delta is the **pure playstyle contribution**, isolated from the retinue effect.

In [ ]:
# Extract KT-twin pairs and compare directly
kt_orig = df_ps_summary[
    (df_ps_summary["retinue"] == "Knight Templar") &
    (df_ps_summary["playstyle"] == "Unshakable")
].copy()

# Twins are named with " (NaturalPS)" suffix
kt_twin = df_ps_summary[df_ps_summary["name"].str.contains(r"\(NaturalPS\)", regex=True)].copy()
print(f"Found {len(kt_orig)} Unshakable KT loadouts and {len(kt_twin)} KT-twins\n")

if len(kt_twin) == 0:
    print("No KT-twins found in summary. Did you re-run the playstyle tournament after adding twins?")
else:
    # Build a join key from equipment (weapon, shield, armor, ranged, tiltyard, tags)
    def equip_key(row):
        return (row.get('weapon', ''), row.get('shield', ''), row.get('armor', ''),
                row.get('ranged', ''), row.get('tiltyard', False), row.get('tags', ''))
    kt_orig['equip_key'] = kt_orig.apply(equip_key, axis=1)
    kt_twin['equip_key'] = kt_twin.apply(equip_key, axis=1)

    merged = kt_orig.merge(
        kt_twin[['equip_key', 'playstyle', 'win_rate', 'decisive_win_rate']],
        on='equip_key', suffixes=('_unshakable', '_natural')
    )
    merged['delta_wr'] = merged['win_rate_unshakable'] - merged['win_rate_natural']
    merged['delta_dec'] = merged['decisive_win_rate_unshakable'] - merged['decisive_win_rate_natural']

    print("=== Pair-level Unshakable vs Natural playstyle (within KT) ===")
    print(merged[['weapon', 'shield', 'armor', 'playstyle_natural',
                  'win_rate_unshakable', 'win_rate_natural', 'delta_wr',
                  'decisive_win_rate_unshakable', 'decisive_win_rate_natural', 'delta_dec']].round(3).to_string(index=False))
    print()
    print(f"=== Summary ===")
    print(f"Mean Δ win rate (Unshakable - Natural): {merged['delta_wr'].mean():+.3f}")
    print(f"Median:                                 {merged['delta_wr'].median():+.3f}")
    print(f"% pairs where Unshakable wins:          {100 * (merged['delta_wr'] > 0).mean():.0f}%")
    print(f"% pairs where Natural wins:             {100 * (merged['delta_wr'] < 0).mean():.0f}%")
    print()
    print(f"Mean Δ decisive win rate:               {merged['delta_dec'].mean():+.3f}")
    print()
    print("INTERPRETATION:")
    print("  Positive delta → Unshakable IS the better playstyle for KT equipment")
    print("  Negative delta → KT does better with the equipment-natural playstyle")
    print("  Near zero      → Playstyle doesn't matter for KT (rout-immunity carries either)")


# 7. Horde Mode — multi-battle survival

Loadouts fight `HORDE_BATTLES` consecutive battles with carry-over fatigue. Tests sustained performance, not single-battle peak.

In [ ]:
if df_horde is None:
    print("Horde data not loaded — skip.")
else:
    print("=== Survival by retinue ===")
    print(df_horde.groupby("retinue").agg(
        n=("name","count"),
        mean_battles=("battles_survived_mean","mean"),
        median_battles=("battles_survived_median","median"),
        size_remaining=("size_remaining_mean","mean"),
    ).round(2).to_string())

    print("\n=== Survival by MPC ===")
    print(df_horde.groupby("mpc").agg(
        n=("name","count"),
        mean_battles=("battles_survived_mean","mean"),
    ).round(2).to_string())

# 8. Loadout-Level Rollups

The detailed per-loadout views: durability, kill power, efficiency, robustness, crush rate, stalemate rate, casualty sources, cause of wipe, shaking, pursuit-presence effects, counters lookup.

## 8.1 Durability — % of battles the loadout's army survives intact

In [ ]:
dur = analysis.durability(df_main_matchups)
print("=== Top 15 most durable ===")
print(dur.head(15).round(3).to_string())

## 8.2 Kill power — average opponents killed per battle

In [ ]:
kp = analysis.kill_power(df_main_matchups)
print("=== Top 15 by kill power ===")
print(kp.head(15).round(3).to_string())

## 8.3 Battle efficiency — wins per 1000 upkeep

In [ ]:
eff = analysis.battle_efficiency(df_main_matchups)
print("=== Top 15 most efficient (wins per 1000 upkeep) ===")
print(eff.head(15).round(3).to_string())

## 8.4 Crush rate — how often this loadout achieves a decisive victory

In [ ]:
cr = analysis.crush_rate(df_main_matchups)
print("=== Top 15 by crush rate ===")
print(cr.head(15).round(3).to_string())

## 8.5 Stalemate rate — how often this loadout's matches go indecisive

In [ ]:
sr = analysis.stalemate_rate(df_main_matchups)
print("=== Top 15 most stalemate-prone ===")
print(sr.head(15).round(3).to_string())

## 8.6 Robustness — consistency of performance across opponents

In [ ]:
rb = analysis.robustness(df_main_matchups)
print("=== Top 15 most robust (consistent across opponents) ===")
print(rb.head(15).round(3).to_string())

## 8.7 Punch above weight — performance vs more expensive opponents

In [ ]:
pa = analysis.punch_above_weight(df_main_matchups)
print("=== Top 15 punch-above-weight ===")
print(pa.head(15).round(3).to_string())

## 8.8 Counters lookup — for a specific loadout, what beats it?

Customize the loadout name to query.

In [ ]:
# Top-winrate loadout by default — change `target` to any name in df_main_summary
target = df_main_summary.nlargest(1, "win_rate")["name"].iloc[0]
print(f"Loadout: {target}\n")
ctr = analysis.counters_for(df_main_matchups, target, top_n=10)
print(f"=== Top 10 counters for {target} ===")
print(ctr.round(3).to_string())

## 8.9 Casualty sources & cause of wipe

What kills armies? Combat strikes, shake tests, or rout?

In [ ]:
# Aggregate across all matchups
cas = df_main_matchups.agg({
    "avg_a_killed_combat": "mean", "avg_a_killed_shake": "mean", "avg_a_killed_rout": "mean",
})
print("=== Average casualties per battle (A's side) ===")
total = cas.sum()
for src, n in cas.items():
    print(f"  {src.replace('avg_a_killed_','').capitalize():<10} {n:.1f}  ({100*n/total:.1f}%)")

# Cause of wipe
wipe_combat = df_main_matchups["a_wipe_combat"].sum()
wipe_shake = df_main_matchups["a_wipe_shake"].sum()
wipe_rout = df_main_matchups["a_wipe_rout"].sum()
total_wipes = wipe_combat + wipe_shake + wipe_rout
if total_wipes > 0:
    print("\n=== Cause of A's wipe (when wiped) ===")
    print(f"  Combat: {wipe_combat:>8}  ({100*wipe_combat/total_wipes:.1f}%)")
    print(f"  Shake:  {wipe_shake:>8}  ({100*wipe_shake/total_wipes:.1f}%)")
    print(f"  Rout:   {wipe_rout:>8}  ({100*wipe_rout/total_wipes:.1f}%)")

## 8.10 Shaking analysis — which loadouts cause / resist shakes?

In [ ]:
# Loadouts whose attacks cause most shake casualties
print("=== Loadouts that inflict the most shake casualties (top 10) ===")
sh = df_main_matchups.groupby("a_name").agg(
    shake_dealt=("avg_b_killed_shake","sum"),
    n_matchups=("a_name","count"),
).reset_index()
sh["shake_per_matchup"] = sh["shake_dealt"] / sh["n_matchups"]
print(sh.nlargest(10, "shake_per_matchup")[["a_name","shake_per_matchup"]].round(2).to_string(index=False))

## 8.11 Pursuit-presence impact

For each pursuit, how much does owning it raise win rate (controlling for MPC)?

In [ ]:
controlled_rows = []
g_p = analysis._pursuit_summary_from_df(df_main_matchups)
all_pursuits = set()
for s in g_p["pursuits"].fillna(""):
    if s: all_pursuits.update(s.split("|"))

for p in sorted(all_pursuits):
    mask = g_p["pursuits"].fillna("").str.contains(rf"(?:^|\|){p}(?:\||$)", regex=True)
    deltas = []
    weights = []
    for mpc in sorted(g_p["military_pursuit_count"].unique()):
        bucket = g_p[g_p["military_pursuit_count"] == mpc]
        with_p = bucket[mask.loc[bucket.index]]
        without_p = bucket[~mask.loc[bucket.index]]
        if len(with_p) >= 3 and len(without_p) >= 3:
            deltas.append(with_p["win_rate"].mean() - without_p["win_rate"].mean())
            weights.append(min(len(with_p), len(without_p)))
    if deltas:
        w = np.array(weights); d = np.array(deltas)
        controlled_rows.append({
            "pursuit": p, "n_buckets": len(deltas),
            "mean_delta_wr": (d*w).sum() / w.sum(),
        })

df_pursuit_effect = pd.DataFrame(controlled_rows).sort_values("mean_delta_wr", ascending=False)
print("=== Pursuit effect on win rate (MPC-controlled) ===")
print(df_pursuit_effect.round(4).to_string(index=False))

## 8.12 Composite dashboard

In [ ]:
print("=== Composite dashboard — top 20 by overall score ===")
dash = analysis.composite_dashboard(df_main_matchups, top_n=20)
print(dash.round(3).to_string())

## 8.13 Cost Analysis — punch-per-upkeep and cheap effective units

The new upkeep-reducing buildings (Levy Hall, Tannery, Armory, Joinery, Master Workshop, Gilded Foundry, ABF, Fletchery, Smokehouse, Butchery) let players field substantially cheaper armies. Which loadouts maximize *effective punch per gold of upkeep*? And which cheap-upkeep units perform well enough to be the backbone of a replenishable army?

In [ ]:
# Build a cost-efficiency dataframe
ce = df_main_summary.copy()
ce['army_size'] = 50  # all loadouts are 50-man

# We measure 3 cost-related metrics:
#   wins_per_1000_upkeep — already in summary
#   wins_per_gold        — same idea, finer granularity
#   replenishment_cost   — gold to replace average casualties = (50 - avg_survivors) * upkeep_per_retinue
ce['wins_per_gold'] = ce['wins'] / ce['army_upkeep'].clip(lower=1)
ce['expected_losses_per_battle'] = (ce['army_size'] - ce['avg_self_survivors']).clip(lower=0)
ce['replenishment_cost_per_battle'] = ce['expected_losses_per_battle'] * ce['upkeep_per_retinue']

# Tag value: gold-efficient with non-trivial win rate
ce['is_cheap_effective'] = (ce['army_upkeep'] <= ce['army_upkeep'].median()) & (ce['win_rate'] >= 0.40)
print(f"Cheap (≤ median upkeep {ce['army_upkeep'].median():.0f}) AND effective (≥40% wr): {ce['is_cheap_effective'].sum()} loadouts")

In [ ]:
# 1) Most efficient gold-per-win loadouts
print("=== Top 20 wins per 1000 upkeep ===")
top_eff = ce.nlargest(20, 'wins_per_1000_upkeep')[
    ['name','retinue','weapon','shield','armor','win_rate','decisive_win_rate',
     'upkeep_per_retinue','army_upkeep','wins_per_1000_upkeep']
]
print(top_eff.round(3).to_string(index=False))

In [ ]:
# 2) Cheap and effective — low upkeep with respectable win rates
print("=== Top 15 'cheap and effective' (low upkeep, decent win rate) ===")
cheap = ce[ce['is_cheap_effective']].copy()
cheap = cheap.nlargest(15, 'win_rate')[
    ['name','retinue','weapon','shield','armor','win_rate','decisive_win_rate',
     'upkeep_per_retinue','army_upkeep','wins_per_1000_upkeep']
]
print(cheap.round(3).to_string(index=False))

In [ ]:
# 3) Lowest replenishment cost — units that don't take heavy losses
print("=== Top 15 lowest replenishment cost per battle (cheap to keep alive) ===")
low_replen = ce[ce['win_rate'] >= 0.35].nsmallest(15, 'replenishment_cost_per_battle')[
    ['name','retinue','weapon','shield','armor','win_rate',
     'avg_self_survivors','upkeep_per_retinue','replenishment_cost_per_battle']
]
print(low_replen.round(2).to_string(index=False))

In [ ]:
# 4) Cost frontier — scatter of upkeep vs win rate, with the Pareto frontier highlighted
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 7))

# Color by retinue
ret_colors = {'Levy':'#888','Man-at-Arms':'#4a90e2','Sergeant':'#d4a800','Knight Templar':'#c43838'}
for ret, color in ret_colors.items():
    sub = ce[ce['retinue'] == ret]
    ax.scatter(sub['army_upkeep'], sub['win_rate'], c=color, alpha=0.4, s=15, label=ret)

# Pareto frontier: for each upkeep bucket, the best win_rate
ce_sorted = ce.sort_values('army_upkeep')
frontier = []
best_wr = -1
for _, row in ce_sorted.iterrows():
    if row['win_rate'] > best_wr:
        best_wr = row['win_rate']
        frontier.append((row['army_upkeep'], row['win_rate'], row['name']))
fx = [f[0] for f in frontier]
fy = [f[1] for f in frontier]
ax.plot(fx, fy, 'k--', linewidth=1.5, alpha=0.6, label='Pareto frontier')

ax.set_xlabel('Army upkeep (gold/turn for 50-man army)')
ax.set_ylabel('Win rate')
ax.set_title('Cost-efficiency frontier: upkeep vs win rate')
ax.legend()
ax.grid(alpha=0.3)
ax.axhline(0.5, color='black', linewidth=0.5, alpha=0.5)
plt.tight_layout(); plt.show()

print(f"\nPareto frontier has {len(frontier)} loadouts. Top of frontier (highest win rate):")
for x, y, name in frontier[-5:]:
    print(f"  upkeep={x:.0f}  wr={y:.3f}  {name[:70]}")

In [ ]:
# 5) Effect of specific upkeep-reducing pursuits on cost efficiency
upkeep_pursuits = ['Levy Hall', 'Tannery', 'Armory', 'Joinery', 'Master Workshop',
                   'Gilded Foundry', 'ABF', 'Fletchery', 'Smokehouse', 'Butchery']

rows = []
for p in upkeep_pursuits:
    mask = ce['pursuits'].fillna('').str.contains(rf"(?:^|\|){p}(?:\||$)", regex=True)
    with_p = ce[mask]
    without_p = ce[~mask]
    if len(with_p) >= 5 and len(without_p) >= 5:
        rows.append({
            'pursuit': p,
            'n_with': len(with_p),
            'n_without': len(without_p),
            'avg_upkeep_with': with_p['army_upkeep'].mean(),
            'avg_upkeep_without': without_p['army_upkeep'].mean(),
            'avg_wr_with': with_p['win_rate'].mean(),
            'avg_wr_without': without_p['win_rate'].mean(),
            'avg_wins_per_1k_with': with_p['wins_per_1000_upkeep'].mean(),
            'avg_wins_per_1k_without': without_p['wins_per_1000_upkeep'].mean(),
        })
df_pursuit_cost = pd.DataFrame(rows)
df_pursuit_cost['Δ_wr'] = df_pursuit_cost['avg_wr_with'] - df_pursuit_cost['avg_wr_without']
df_pursuit_cost['Δ_efficiency'] = df_pursuit_cost['avg_wins_per_1k_with'] - df_pursuit_cost['avg_wins_per_1k_without']
print("=== Impact of upkeep-reducing pursuits ===")
print(df_pursuit_cost.round(3).to_string(index=False))
print()
print("Δ_wr near zero means: the pursuit doesn't change which loadouts win, just makes them cheaper.")
print("Δ_efficiency positive means: the pursuit makes loadouts more cost-efficient (more wins per gold).")

In [ ]:
# 6) Replenishment scenarios — given a fixed reserve, how many battles can each loadout fight?
RESERVE_GOLD = 5000   # adjust to test different reserve budgets

ce['battles_per_reserve'] = RESERVE_GOLD / ce['replenishment_cost_per_battle'].clip(lower=1)
ce['effective_wins_per_reserve'] = ce['battles_per_reserve'] * ce['win_rate']

print(f"=== Given a {RESERVE_GOLD} gold reserve, expected wins before reinforcement gold runs out ===")
top_reserve = ce[ce['win_rate'] >= 0.30].nlargest(15, 'effective_wins_per_reserve')[
    ['name','retinue','weapon','win_rate','replenishment_cost_per_battle',
     'battles_per_reserve','effective_wins_per_reserve']
]
print(top_reserve.round(2).to_string(index=False))

# 9. Headline Findings — Design Targets

The key design target was: **win rate tightly correlates with Military Pursuit Count.** Restate and evaluate.

In [ ]:
print("=" * 60)
print("DESIGN TARGET CHECK")
print("=" * 60)
print()
print(f"1. MPC ↔ Win Rate correlation:")
print(f"   Pearson r  = {pearson_r:+.3f}  (target: > +0.7 for strong correlation)")
print(f"   Spearman r = {spearman_r:+.3f}")
print()
print(f"2. Higher MPC beats lower MPC?")
print(f"   Average win rate vs lower-MPC opponents: {df_vs['vs_lower_mean'].mean():.3f}")
print(f"   Average win rate vs higher-MPC opponents: {df_vs['vs_higher_mean'].mean():.3f}")
print(f"   Difference: {df_vs['vs_lower_mean'].mean() - df_vs['vs_higher_mean'].mean():+.3f}")
print()
print(f"3. Domain Count independent effect:")
print(f"   Pearson r (raw)  = {pearson_d:+.3f}")
print(f"   Residual r (after MPC removed) = {res_pearson:+.3f}")
print()
print(f"4. Retinue win-rate spread:")
print(f"   Best retinue:  {ret_summary['win_rate'].idxmax():<15} {ret_summary['win_rate'].max():.3f}")
print(f"   Worst retinue: {ret_summary['win_rate'].idxmin():<15} {ret_summary['win_rate'].min():.3f}")
print(f"   Spread:        {ret_summary['win_rate'].max() - ret_summary['win_rate'].min():.3f}")
print()
print(f"5. Decisive vs stalemate balance:")
print(f"   Avg decisive_rate: {df_main_summary['decisive_rate'].mean():.3f}")
print(f"   (target: > 0.7 for meaningful battles)")